In [65]:
import os

# --- ！！！必须放在第一行！！！ ---
# 允许 OpenMP 库的多个副本共存。这是解决 Torch 和 Scipy 冲突的万能药。
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

## using "ln -sf /usr/lib/x86_64-linux-gnu/libstdc++.so.6 ${CONDA_PREFIX}/lib/libstdc++.so.6" if open3d fails in UBUNTU

In [66]:
import random
import torch
import numpy as np
def setup_seed(seed: int = 42, deterministic: bool = False):
    """
    统一设置 PyTorch/NumPy/Random 的随机种子
    @param seed: 基础随机种子（建议 0-2^32-1 之间）
    @param deterministic: 是否开启严格确定性模式（可能影响 GPU 性能）
    """
    # Python 内置随机数
    random.seed(seed)
    
    # NumPy 随机性
    np.random.seed(seed)
    
    # PyTorch CPU 随机性
    torch.manual_seed(seed)
    
    # PyTorch GPU 随机性（单卡+多卡）
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # 多卡情况
    
    # 确定性优化（需谨慎！可能降低 GPU 速度）
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.enabled = False  # 如需使用 cudnn，建议仅设置 deterministic=True
        
setup_seed(seed=42)

In [67]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import glob
import os
from tqdm import tqdm
import random
from sklearn.cluster import MiniBatchKMeans
import gc  # [NEW] Explicit garbage collection

# ==========================================
# 1. Math Tools
# ==========================================
def axis_angle_to_rotation_6d(axis_angle):
    """Convert axis-angle (..., 3) to continuous 6D rotation (..., 6)."""
    theta = torch.norm(axis_angle, dim=-1, keepdim=True)
    eps = 1e-6
    k = axis_angle / (theta + eps)
    x, y, z = k[..., 0], k[..., 1], k[..., 2]
    zeros = torch.zeros_like(x)
    K = torch.stack([zeros, -z, y, z, zeros, -x, -y, x, zeros], dim=-1).view(axis_angle.shape[:-1] + (3, 3))
    I = torch.eye(3, device=axis_angle.device).expand_as(K)
    R = I + torch.sin(theta).unsqueeze(-1) * K + (1 - torch.cos(theta)).unsqueeze(-1) * (K @ K)
    return R[..., :2].flatten(-2)

# ==========================================
# 2. Dataset Class (SAFE Mode + Optimized)
# ==========================================
class AMASSSequenceDataset(Dataset):
    def __init__(self, file_list, 
                 split_type='train', 
                 val_ratio=0.3,
                 total_frames=91,     # Prediction Window T
                 sparse_step=10,      # Observation Interval h
                 history_k=10,        # History Length k
                 condition_sec=3.0):
        
        self.files = file_list
        self.total_frames = total_frames
        self.sparse_step = sparse_step
        self.history_k = history_k
        
        # Calculate window sizes
        self.history_frames = self.history_k * self.sparse_step
        self.window_size_needed = self.history_frames + self.total_frames
        
        self.indices = []
        if split_type == 'train':
            stride = sparse_step
        else:
            stride = total_frames-1
        # Indexing with Safe Split (Within-File)
        for f_path in self.files:
            try:
                # [OPTIONAL] 'mmap_mode' could be used here for massive files, but standard load is safer for stability
                with np.load(f_path, allow_pickle=True) as data:
                    if 'poses' not in data: continue
                    n_samples = data['poses'].shape[0]
                    
                    if n_samples <= self.window_size_needed: continue
                    
                    split_frame_idx = int(n_samples * (1 - val_ratio))
                    
                    valid_starts = []
                    
                    if split_type == 'train':
                        end_limit = split_frame_idx - self.window_size_needed
                        if end_limit > 0:
                            valid_starts = range(0, end_limit + 1, stride)
                            
                    else:
                        start_limit = split_frame_idx
                        end_limit = n_samples - self.window_size_needed
                        if end_limit > start_limit:
                            valid_starts = range(start_limit, end_limit + 1, stride)

                    for start_idx in valid_starts:
                        self.indices.append((f_path, start_idx))
            except: continue

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        f_path, history_start = self.indices[idx]
        try:
            # [OPTIMIZATION] Load directly to float32 to avoid float64 overhead if applicable
            data = np.load(f_path, allow_pickle=True)
            load_start = history_start
            load_end = history_start + self.window_size_needed
            
            # 1. Load Raw Data
            trans_raw = torch.from_numpy(data['trans'][load_start : load_end]).float()
            poses_raw = torch.from_numpy(data['poses'][load_start : load_end, :66]).float()
            
            if 'markers_sim' in data:
                markers_raw = torch.from_numpy(data['markers_sim'][load_start : load_end]).float()
            elif 'markers' in data:
                markers_raw = torch.from_numpy(data['markers'][load_start : load_end]).float()
            else:
                return self.__getitem__(random.randint(0, len(self)-1))

            if trans_raw.shape[0] != self.window_size_needed:
                 return self.__getitem__(random.randint(0, len(self)-1))

            # 2. Normalize Root Translation
            root_bias = trans_raw[0].clone()
            trans_norm = trans_raw - root_bias
            
            # Prepare Full State (Dense)
            B = poses_raw.shape[0]
            poses_6d = axis_angle_to_rotation_6d(poses_raw.view(B, 22, 3)).view(B, -1)
            full_seq_dense = torch.cat([trans_norm, poses_6d], dim=-1) 

            # 3. Construct VR Condition (Pos + Vel)
            vr_idx = [24, 40, 6] 
            
            # Calculate Scale (Z-up assumption)
            ref_idx = self.history_frames
            h = markers_raw[ref_idx, 24, 2] - markers_raw[ref_idx, 27, 2]
            scale = 1.0 / h if h > 0.1 else 0.6
            
            # 3.1 VR Position (Local)
            vr_pos_global = markers_raw[:, vr_idx, :]
            vr_pos_local = (vr_pos_global - trans_norm.unsqueeze(1)) * scale
            
            # 3.2 VR Velocity (Backward Difference)
            vr_vel_local = torch.zeros_like(vr_pos_local)
            vr_vel_local[1:] = (vr_pos_local[1:] - vr_pos_local[:-1]) * 30.0 # m/s
            
            # 3.3 Root Velocity (Global)
            vel_root = torch.zeros_like(trans_norm)
            vel_root[1:] = (trans_norm[1:] - trans_norm[:-1]) * 30.0
            vel_root = vel_root * scale

            # 4. Slice for Prediction Window
            pred_slice = slice(self.history_frames, self.history_frames + self.total_frames)
            
            curr_vr_pos = vr_pos_local[pred_slice]   # (T_pred, 3, 3)
            curr_vr_vel = vr_vel_local[pred_slice]   # (T_pred, 3, 3)
            curr_root_v = vel_root[pred_slice]       # (T_pred, 3)
            
            # 5. Sparse Sampling for Condition
            sp_idx = torch.arange(0, self.total_frames, self.sparse_step)
            # [FIX] Ensure index doesn't exceed T_FRAMES even with float arithmetic issues
            sp_idx = sp_idx[sp_idx < self.total_frames]
            
            # Final Condition
            cond_matrix = torch.cat([
                curr_root_v[sp_idx], 
                curr_vr_pos[sp_idx].view(-1, 9),
                curr_vr_vel[sp_idx].view(-1, 9)
            ], dim=-1)
            
            # 6. Construct Target (Takens' Augmented State)
            # [OPTIMIZATION] Vectorized Construction
            # Instead of a Python loop appending to a list, we use slicing.
            # We want: [t, t-h, t-2h, ..., t-kh]
            # full_seq_dense contains [History .... Prediction]
            
            delayed_views = []
            for k in range(self.history_k + 1):
                # k=0 -> t (current)
                # k=1 -> t-h (previous)
                # offset determines the start position relative to history_frames
                offset = k * self.sparse_step
                start_i = self.history_frames - offset
                end_i = start_i + self.total_frames
                
                # Append slice (view)
                delayed_views.append(full_seq_dense[start_i : end_i])
            
            # Your original code used range(history_k, -1, -1), meaning [Oldest, ..., Newest].
            # My loop above generates [Newest (t), ..., Oldest (t-kh)].
            # We reverse to match original behavior:
            delayed_views.reverse()
            
            target_dense = torch.cat(delayed_views, dim=-1) # (T_pred, D_tilde)
            target_sparse = target_dense[sp_idx]

            return {
                'condition': cond_matrix,        # (N_sparse, 21)
                'target_sparse': target_sparse,  # (N_sparse, D_tilde)
                'target_dense': target_dense,    # (T_pred, D_tilde)
                't_sparse': sp_idx.float() / 30.0,
                't_dense': torch.arange(self.total_frames).float() / 30.0
            }
        except:
            return self.__getitem__(random.randint(0, len(self)-1))

# ==========================================
# 3. Ambiguity-Aware Filtering Logic (Unchanged)
# ==========================================
def get_file_feature(f_path, win_len=91):
    try:
        data = np.load(f_path, allow_pickle=True)
        if 'poses' not in data or 'trans' not in data: return None
        win_len = (win_len // 15)+1
        poses = data['poses'][::15]
        trans = data['trans'][::15]
        T = poses.shape[0]
        if T <= win_len: return None
        
        # --- A. Calculate Pose Energy Score ---
        pose_vel = np.linalg.norm(poses[1:] - poses[:-1], axis=1) 
        window = np.ones(win_len) / win_len
        energy_profile = np.convolve(pose_vel, window, mode='valid')
        
        best_start = np.argmax(energy_profile)
        best_score = energy_profile[best_start]
        
        if best_score < 0.01: return None
        
        # --- B. Extract VR Input Feature ---
        best_end = best_start + win_len
        
        if 'markers_sim' in data: markers = data['markers_sim'][::15]
        elif 'markers' in data: markers = data['markers'][::15]
        else: return None
        
        m_win = markers[best_start:best_end] 
        t_win = trans[best_start:best_end]   
        
        vr_idx = [24, 40, 6]
        h = m_win[0, 24, 2] - m_win[0, 27, 2]
        scale = 1.0 / h if h > 0.1 else 0.6
        
        vr_local = (m_win[:, vr_idx, :] - t_win[:, None, :]) * scale
        feature_vector = vr_local.flatten() 
        
        return {
            'f_path': f_path,
            'feature': feature_vector, 
            'score': best_score        
        }
        
    except Exception:
        return None

def cluster_and_filter_files(file_list, target_count=150):
    print(f"\n--- Filtering {len(file_list)} files via Input-Clustering (Target: {target_count}) ---")
    
    metas = []
    for f in tqdm(file_list, desc="Analyzing Ambiguity"):
        m = get_file_feature(f)
        if m: metas.append(m)
        
    if not metas: return []
    print(f"    Valid candidate files: {len(metas)}")
    
    X = np.stack([m['feature'] for m in metas])
    n_clusters = min(target_count, len(metas))
    
    print(f"    Clustering Inputs (k={n_clusters})...")
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=256, n_init='auto', random_state=42).fit(X)
    labels = kmeans.labels_
    
    final_files = []
    clusters = {}
    for i, lbl in enumerate(labels):
        if lbl not in clusters: clusters[lbl] = []
        clusters[lbl].append(metas[i])
        
    for lbl in clusters:
        winner = max(clusters[lbl], key=lambda x: x['score'])
        final_files.append(winner['f_path'])
        
    print(f"    Selected {len(final_files)} representative files (Ambiguity Resolved).")
    return final_files

# ==========================================
# 4. Generator Workflow (Memory Optimized)
# ==========================================
def generate_dataset_tensors(data_root, output_path, 
                             samples_needed=8192, 
                             max_scan_files=3000,
                             target_clusters=150, 
                             total_frames=91,
                             sparse_step=10,
                             history_k=10,
                             clustering=True,
                             viz_target_name="144_10_stageii.npz"): 
    
    print(f"\n=== Dataset Generation: Ambiguity-Resolved & Safe-Split ===")
    
    # 1. Scan Files
    all_files = glob.glob(os.path.join(data_root, '**/*.npz'), recursive=True)
    valid_files = [f for f in all_files if 'stageii' in os.path.basename(f)]
    
    vip_file = None
    pool_files = []
    
    for f in valid_files:
        if viz_target_name in f:
            vip_file = f
            print(f" [VIP] Found Visualization Target: {os.path.basename(f)}")
        else:
            pool_files.append(f)
            
    if vip_file is None:
        print(f" [Warning] Visualization target '{viz_target_name}' not found in path.")

    # Shuffle the rest
    random.seed(42)
    random.shuffle(pool_files)
    pool_files = pool_files[:max_scan_files]
    
    # 2. Filter the POOL
    if clustering:
        filtered_pool = cluster_and_filter_files(pool_files, target_count=target_clusters)
    else:
        filtered_pool = pool_files
    
    if not filtered_pool:
        print("Error: No files selected from pool.")
        return

    # 3. Merge
    if vip_file:
        final_files = [vip_file] + filtered_pool
    else:
        final_files = filtered_pool

    ds_kwargs = dict(total_frames=total_frames, sparse_step=sparse_step, history_k=history_k)
    
    train_ds = AMASSSequenceDataset(final_files, split_type='train', **ds_kwargs)
    val_ds   = AMASSSequenceDataset(final_files, split_type='val', **ds_kwargs)
    
    # 4. Extract Tensors
    output_data = {}
    
    # [OPTIMIZATION] Pre-allocate tensors instead of using lists
    for split, ds in zip(['train', 'val'], [train_ds, val_ds]):
        if len(ds) == 0: continue
        
        limit = min(samples_needed, len(ds))
        loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)
        
        # Probe first item for shapes
        sample = ds[0]
        dim_cond = sample['condition'].shape
        dim_sparse = sample['target_sparse'].shape
        dim_dense = sample['target_dense'].shape
        
        # Pre-allocate large tensors
        print(f"Allocating Memory for {split} ({limit} samples)...")
        out_cond = torch.zeros((limit, *dim_cond), dtype=torch.float32)
        
        if split == 'train':
            out_traj = torch.zeros((limit, *dim_sparse), dtype=torch.float32)
        else:
            # Val often stores dense trajectory
            out_traj = torch.zeros((limit, *dim_dense), dtype=torch.float32)
            
        cnt = 0
        
        print(f"Extracting {split} tensors...")
        for batch in tqdm(loader):
            B = batch['condition'].shape[0]
            
            # Determine indices
            end_idx = cnt + B
            if end_idx > limit:
                # Truncate batch if we exceed limit
                B = limit - cnt
                end_idx = limit
                batch['condition'] = batch['condition'][:B]
                if split == 'train':
                    batch['target_sparse'] = batch['target_sparse'][:B]
                else:
                    batch['target_dense'] = batch['target_dense'][:B]

            # Fill in the pre-allocated tensor
            out_cond[cnt : end_idx] = batch['condition']
            
            if split == 'train':
                out_traj[cnt : end_idx] = batch['target_sparse']
            else:
                out_traj[cnt : end_idx] = batch['target_dense']
            
            cnt = end_idx
            if cnt >= limit: break
        
        # Save to dictionary
        if split == 'train':
            output_data['train_data'] = {
                'conditions': out_cond,
                'trajectories': out_traj,
                'observation_times': ds[0]['t_sparse'],
                'check_info': 'Sparse Target (x_tilde)'
            }
        else:
            output_data['val_data'] = {
                'conditions': out_cond,
                'trajectories': out_traj,
                'observation_times': ds[0]['t_dense'],
                'check_info': 'Dense Target (x_tilde)'
            }
            
        print(f"  -> {split} saved: {out_cond.shape[0]} samples")
        
        # [OPTIMIZATION] Explicit Garbage Collection
        # Important: Free the huge pre-allocated tensors before starting next split
        del out_cond, out_traj, loader
        gc.collect()

    torch.save(output_data, output_path)
    print(f"Done. Saved to {output_path}")

# ==========================================
# 5. Main Entry (Modified for Memory Safety)
# ==========================================
import gc # Ensure gc is imported

if __name__ == "__main__":
    # Config
    ROOT = './data/CMU' 
    OUT = 'amass_processed_ambiguity_free.pt'
    
    # Params matching your SDE model requirements
    T_FRAMES = 91        # 3s * 30fps
    SPARSE_STEP = 15     # h
    HISTORY_K = 5        # x_tilde dim multiplier
    CLUSTERS = 100       # Number of distinct input patterns to keep
    
    # 1. Generate
    if os.path.exists(ROOT):
        # Run generation
        generate_dataset_tensors(ROOT, OUT, 
                                 samples_needed=400000, 
                                 max_scan_files=10000,
                                 target_clusters=CLUSTERS,
                                 total_frames=T_FRAMES,
                                 sparse_step=SPARSE_STEP,
                                 history_k=HISTORY_K)
        
        # [MEMORY FIX] Explicitly clear memory after generation
        print("\n[System] Cleaning up memory before verification...")
        gc.collect()            # Force Python garbage collector
        torch.cuda.empty_cache() # If GPU was touched, clear cache
        
        # Optional: Sleep briefly to let OS reclaim memory
        import time
        time.sleep(2)

    # 2. Verify
    if os.path.exists(OUT):
        print("\n=== VERIFICATION ===")
        
        try:
            # [MEMORY FIX] Load with explicit map_location to avoid GPU spikes
            # If 14GB is still too tight, we can't do much because torch.load reads the whole dict.
            # But the gc.collect() above should solve the fragmentation.
            data = torch.load(OUT, map_location='cpu')
            
            exp_dim = (HISTORY_K + 1) * 135 
            
            for k in ['train_data', 'val_data']:
                if k in data:
                    # Access tensor
                    traj = data[k]['trajectories']
                    
                    # Check dimensions
                    current_dim = traj.shape[-1]
                    status = 'OK' if current_dim == exp_dim else 'FAIL'
                    
                    print(f"{k}: {traj.shape} | Expected Dim: {exp_dim} | Status: {status}")
                    
                    # [MEMORY FIX] Delete immediately after checking to free space for next key
                    del traj
                    
            # Final cleanup
            # del data
            gc.collect()
            print("\nVerification Complete.")
            
        except RuntimeError as e:
            print(f"\n[Error] Could not load file for verification due to memory limits.")
            print(f"System Message: {e}")
            print("Note: The file is likely saved correctly, but your RAM (32GB) is full from the generation step.")
            print("Try running a separate script just for verification.")


=== Dataset Generation: Ambiguity-Resolved & Safe-Split ===
 [VIP] Found Visualization Target: 144_10_stageii.npz

--- Filtering 1982 files via Input-Clustering (Target: 100) ---


Analyzing Ambiguity: 100%|██████████| 1982/1982 [00:29<00:00, 66.61it/s]
c:\ProgramData\anaconda3\envs\csi\lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 5120 or by setting the environment variable OMP_NUM_THREADS=1
  warnings.warn(


    Valid candidate files: 1745
    Clustering Inputs (k=100)...
    Selected 100 representative files (Ambiguity Resolved).
Allocating Memory for train (5713 samples)...
Extracting train tensors...


 99%|█████████▉| 89/90 [00:52<00:00,  1.71it/s]


  -> train saved: 5713 samples
Allocating Memory for val (360 samples)...
Extracting val tensors...


 83%|████████▎ | 5/6 [00:03<00:00,  1.40it/s]


  -> val saved: 360 samples
Done. Saved to amass_processed_ambiguity_free.pt

[System] Cleaning up memory before verification...

=== VERIFICATION ===
train_data: torch.Size([5713, 7, 810]) | Expected Dim: 810 | Status: OK
val_data: torch.Size([360, 91, 810]) | Expected Dim: 810 | Status: OK

Verification Complete.


In [68]:
import torch
import torch.nn.functional as F
import os
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

import cubic_SI
from cubic_SI import model
from cubic_SI import computations as cmp

train_data = data.get('train_data')
val_data = data.get('val_data')

In [69]:
SI_class = model.Cubic_SI_model(train_data['trajectories'], train_data['observation_times'], condition_tensor=train_data['conditions'],
                                n_layers=8,hiden_size=1024,
                                N_training=50000, B=1024, model_lr=1e-4,steps=train_data['observation_times'][-1] * 120,
                                # func_type='sqrt',
                                # u_t=lambda x: 1e-2,
                                spline=True,
                                save_path='model_history/mocap_spline',record_gap=10 #,C_d=32,use_conditional_path_encoder=True,hist_times=train_data['condition_times']
                                ,use_mlp=False,dynamic_conditions=True
                                )
# SI_class.train()
SI_class.model_load('model_history/mocap_spline/model.pt')

generated_paths_ODE = SI_class.eval(val_data['trajectories'][:,0,:],conditions=val_data['conditions'],SDE=False)
generated_paths_ODE = torch.stack(generated_paths_ODE).transpose(0,1)
generated_paths_ODE_spline = generated_paths_ODE
del generated_paths_ODE

In [70]:
SI_class = model.Cubic_SI_model(train_data['trajectories'], train_data['observation_times'], condition_tensor=train_data['conditions'],
                                n_layers=8,hiden_size=1024,
                                N_training=50000, B=1024, model_lr=1e-4,steps=train_data['observation_times'][-1] * 120,
                                # func_type='sqrt',
                                # u_t=lambda x: 1e-2,
                                spline=False,
                                save_path='model_history/mocap_linear',record_gap=10 #,C_d=32,use_conditional_path_encoder=True,hist_times=train_data['condition_times']
                                ,use_mlp=False,dynamic_conditions=True
                                )
# SI_class.train()
SI_class.model_load('model_history/mocap_linear/model.pt')

generated_paths_ODE = SI_class.eval(val_data['trajectories'][:,0,:],conditions=val_data['conditions'],SDE=False)
generated_paths_ODE = torch.stack(generated_paths_ODE).transpose(0,1)
generated_paths_ODE_linear = generated_paths_ODE
del generated_paths_ODE

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
from pathlib import Path
import smplx
import open3d as o3d
import imageio

# ==============================================================================
# 1. Math Utilities
# ==============================================================================

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    """ Convert continuous 6D rotation (..., 6) to rotation matrix (..., 3, 3). """
    d6_reshaped = d6.view(*d6.shape[:-1], 3, 2)
    a1 = d6_reshaped[..., 0]
    a2 = d6_reshaped[..., 1]
    b1 = F.normalize(a1, dim=-1)
    b2 = a2 - (b1 * a2).sum(-1, keepdim=True) * b1
    b2 = F.normalize(b2, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack((b1, b2, b3), dim=-1)

def robust_matrix_to_axis_angle(matrix):
    """ Convert rotation matrix (..., 3, 3) to axis-angle (..., 3). Robust to singularities. """
    if not torch.is_tensor(matrix): matrix = torch.tensor(matrix)
    m00, m11, m22 = matrix[..., 0, 0], matrix[..., 1, 1], matrix[..., 2, 2]
    trace = m00 + m11 + m22
    cos = 0.5 * (trace - 1)
    cos = torch.clamp(cos, -1 + 1e-6, 1 - 1e-6)
    theta = torch.acos(cos)
    
    m21, m12 = matrix[..., 2, 1], matrix[..., 1, 2]
    m02, m20 = matrix[..., 0, 2], matrix[..., 2, 0]
    m10, m01 = matrix[..., 1, 0], matrix[..., 0, 1]
    
    sin = torch.sin(theta)
    denom = 2 * sin + 1e-6
    
    axis0 = (m21 - m12) / denom
    axis1 = (m02 - m20) / denom
    axis2 = (m10 - m01) / denom
    
    axis_angle = torch.stack([axis0, axis1, axis2], dim=-1) * theta.unsqueeze(-1)
    mask = theta < 1e-4
    axis_angle[mask] = 0.0
    return axis_angle

# ==============================================================================
# 2. Fast Forward Kinematics (Singleton)
# ==============================================================================

class FastRigidFK(torch.nn.Module):
    def __init__(self, model_path, device):
        super().__init__()
        self.device = device
        print(f"[FastRigidFK] Loading SMPL-X from {model_path}...")
        
        # --- SAFE INIT: Load on CPU first to catch errors early ---
        # This prevents "Device-side assert" if SMPL fails to load,
        # and prevents VRAM OOM during initialization.
        try:
            temp_model = smplx.create(model_path, model_type='smplx', gender='neutral', 
                                      use_pca=False, batch_size=1)
        except Exception as e:
            raise RuntimeError(f"Failed to load SMPL model at {model_path}. Error: {e}")

        # Extract kinematic tree (CPU)
        parents = temp_model.parents[:22].clone()
        
        # Run dry pass to get rest pose (CPU)
        zero_betas = torch.zeros(1, 10)
        zero_trans = torch.zeros(1, 3)
        output = temp_model(betas=zero_betas, transl=zero_trans)
        J_rest = output.joints[0, :22, :] 
        
        # --- TRANSFER: Only move necessary buffers to GPU ---
        self.register_buffer('parents', parents.to(device))
        self.register_buffer('J_rest', J_rest.to(device))
        
        # Calculate bone offsets
        local_offsets = J_rest.clone()
        for i in range(1, 22):
            parent_idx = self.parents[i]
            local_offsets[i] = J_rest[i] - J_rest[parent_idx]
            
        self.register_buffer('local_offsets', local_offsets.to(device))
        
        # Cleanup CPU model
        del temp_model, output
        print("[FastRigidFK] Skeleton extracted successfully.")

    def forward(self, rot_mats, trans):
        # (Same as before)
        B = rot_mats.shape[0]
        glob_transforms = torch.eye(4, device=self.device).reshape(1, 1, 4, 4).repeat(B, 22, 1, 1)
        local_transforms = torch.eye(4, device=self.device).reshape(1, 1, 4, 4).repeat(B, 22, 1, 1)
        
        local_transforms[..., :3, :3] = rot_mats
        local_transforms[..., :3, 3] = self.local_offsets.unsqueeze(0)
        
        glob_transforms[:, 0] = local_transforms[:, 0]
        glob_transforms[:, 0, :3, 3] += trans
        
        for i in range(1, 22):
            parent_idx = self.parents[i]
            glob_transforms[:, i] = torch.matmul(glob_transforms[:, parent_idx], local_transforms[:, i])
            
        return glob_transforms[..., :3, 3]
    
# ==============================================================================
# 3. Advanced Metric Evaluation (MPJPE, MPJVE, Jitter)
# ==============================================================================

def evaluate_motion_quality(generated_tensor, true_tensor, device, smpl_path='./data', batch_size=1024):
    """
    Memory Safe Evaluation:
    - Inputs: CPU Tensors
    - Processing: Moves small batches to GPU -> Calc -> Move result to CPU
    - Output: Lists of metrics
    """
    # 1. Singleton FK Loader
    if not hasattr(evaluate_motion_quality, 'fast_fk'):
        if not os.path.exists(smpl_path):
            print(f"[Warning] SMPL path {smpl_path} not found. Metrics cannot be computed.")
            return {'MPJPE': [], 'MPJVE': [], 'Jitter': []}
        evaluate_motion_quality.fast_fk = FastRigidFK(smpl_path, device)
    
    fk_layer = evaluate_motion_quality.fast_fk
    
    # 2. Data trimming (CPU side)
    if generated_tensor.shape[-1] != 135:
        generated_tensor = generated_tensor[..., -135:]
        true_tensor = true_tensor[..., -135:]
        
    T = min(generated_tensor.shape[1], true_tensor.shape[1])
    gen_seq = generated_tensor[:, :T, :]
    true_seq = true_tensor[:, :T, :]
    B_seq = gen_seq.shape[0]
    
    # Flatten everything (CPU)
    gen_flat = gen_seq.reshape(-1, 135)
    true_flat = true_seq.reshape(-1, 135)
    total_frames = gen_flat.shape[0]
    
    # 3. Batched GPU Processing
    gen_joints_cpu = []
    true_joints_cpu = []
    
    # No gradients needed for evaluation
    with torch.no_grad():
        for i in range(0, total_frames, batch_size):
            end = min(i + batch_size, total_frames)
            
            # A. Move ONLY this batch to GPU
            g_batch = gen_flat[i:end].to(device)
            t_batch = true_flat[i:end].to(device)
            
            # B. Run FK
            g_rmat = rotation_6d_to_matrix(g_batch[:, 3:].reshape(-1, 22, 6))
            g_joints = fk_layer(g_rmat, g_batch[:, :3])
            
            t_rmat = rotation_6d_to_matrix(t_batch[:, 3:].reshape(-1, 22, 6))
            t_joints = fk_layer(t_rmat, t_batch[:, :3])
            
            # C. Move results to CPU and clear GPU memory immediately
            gen_joints_cpu.append(g_joints.cpu())
            true_joints_cpu.append(t_joints.cpu())
            
            del g_batch, t_batch, g_rmat, t_rmat, g_joints, t_joints
    
    # 4. Compute Metrics on CPU (Cheaper and safer)
    # Reassemble (B, T, 22, 3)
    gen_joints = torch.cat(gen_joints_cpu, dim=0).reshape(B_seq, T, 22, 3)
    true_joints = torch.cat(true_joints_cpu, dim=0).reshape(B_seq, T, 22, 3)
    
    # MPJPE
    gen_rel = gen_joints - gen_joints[:, :, 0:1, :]
    true_rel = true_joints - true_joints[:, :, 0:1, :]
    mpjpe_seq = torch.norm(gen_rel - true_rel, dim=-1).mean(dim=-1) * 1000.0

    # MPJVE
    gen_vel = gen_joints[:, 1:] - gen_joints[:, :-1]
    true_vel = true_joints[:, 1:] - true_joints[:, :-1]
    mpjve_seq = torch.norm(gen_vel - true_vel, dim=-1).mean(dim=-1) * 1000.0
    mpjve_seq = torch.cat([torch.zeros(B_seq, 1), mpjve_seq], dim=1)

    # Jitter
    gen_acc = gen_vel[:, 1:] - gen_vel[:, :-1]
    gen_jerk = gen_acc[:, 1:] - gen_acc[:, :-1]
    jitter_seq = torch.norm(gen_jerk, dim=-1).mean(dim=-1) * 1000.0
    jitter_seq = torch.cat([torch.zeros(B_seq, 3), jitter_seq], dim=1)
    
    return {
        'MPJPE': mpjpe_seq.mean(dim=0).tolist(),
        'MPJVE': mpjve_seq.mean(dim=0).tolist(),
        'Jitter': jitter_seq.mean(dim=0).tolist()
    }


def compute_batch_metrics(gen_batch, true_batch, fk_layer):
    """
    Computes metrics for a single batch on the GPU.
    Returns the sum of errors over the batch (to be averaged later).
    """
    B, T, _ = gen_batch.shape
    
    # 1. Dimension Check
    if gen_batch.shape[-1] != 135: gen_batch = gen_batch[..., -135:]
    if true_batch.shape[-1] != 135: true_batch = true_batch[..., -135:]

    # 2. Forward Kinematics (On GPU)
    g_flat = gen_batch.reshape(-1, 135)
    t_flat = true_batch.reshape(-1, 135)

    g_rmat = rotation_6d_to_matrix(g_flat[:, 3:].reshape(-1, 22, 6))
    t_rmat = rotation_6d_to_matrix(t_flat[:, 3:].reshape(-1, 22, 6))

    g_joints = fk_layer(g_rmat, g_flat[:, :3]).reshape(B, T, 22, 3)
    t_joints = fk_layer(t_rmat, t_flat[:, :3]).reshape(B, T, 22, 3)

    # --- 3. MPJPE (Position) ---
    g_rel = g_joints - g_joints[:, :, 0:1, :]
    t_rel = t_joints - t_joints[:, :, 0:1, :]
    # Shape: (B, T)
    mpjpe_batch = torch.norm(g_rel - t_rel, dim=-1).mean(dim=-1) * 1000.0

    # --- 4. MPJVE (Velocity) ---
    # Velocity: (B, T-1, 22, 3)
    g_vel = g_joints[:, 1:] - g_joints[:, :-1]
    t_vel = t_joints[:, 1:] - t_joints[:, :-1]
    
    # Error: (B, T-1)
    mpjve_batch_valid = torch.norm(g_vel - t_vel, dim=-1).mean(dim=-1) * 1000.0
    # Pad first frame with 0 to match T (same as original code)
    zeros = torch.zeros(B, 1, device=gen_batch.device)
    mpjve_batch = torch.cat([zeros, mpjve_batch_valid], dim=1)

    # --- 5. Jitter (Naturalness) ---
    # Acceleration: (B, T-2, 22, 3)
    g_acc = g_vel[:, 1:] - g_vel[:, :-1]
    # Jerk: (B, T-3, 22, 3)
    g_jerk = g_acc[:, 1:] - g_acc[:, :-1]
    
    # Magnitude: (B, T-3)
    jitter_batch_valid = torch.norm(g_jerk, dim=-1).mean(dim=-1) * 1000.0
    # Pad first 3 frames with 0
    zeros_3 = torch.zeros(B, 3, device=gen_batch.device)
    jitter_batch = torch.cat([zeros_3, jitter_batch_valid], dim=1)

    # Return Sum over Batch (B dim) so we can average later
    return {
        'MPJPE_Sum': mpjpe_batch.sum(dim=0).cpu(), # (T,)
        'MPJVE_Sum': mpjve_batch.sum(dim=0).cpu(), # (T,)
        'Jitter_Sum': jitter_batch.sum(dim=0).cpu() # (T,)
    }

# ==============================================================================
# 4. Visualization Tools
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

import seaborn as sns
import matplotlib.pyplot as plt

def plot_comparative_metrics(results_dict, times, title_prefix, color_map):
    """
    (Unified Style / 统一风格版)
    
    1. 遍历 results_dict 中的每一个 metric。
    2. 为每个 metric 生成一张独立的 PDF。
    3. 风格：白底、灰网格、全黑边框 (Boxed)、无标题。
    """
    
    # --- 修改点 1：统一使用 whitegrid 和字体设置 ---
    sns.set_theme(style="whitegrid", rc={
        "font.family": "serif",      # 衬线体
        "axes.edgecolor": "black",   # 边框黑色
        "axes.linewidth": 1.0,       # 边框线宽
        "grid.color": "#E0E0E0",     # 网格颜色
        "legend.frameon": True,      # 图例带框
        "legend.edgecolor": "black", # 图例边框黑色
        "legend.fancybox": False     # 图例直角
    })
    
    metrics = list(results_dict.keys())
    print(f"Plotting {len(metrics)} metrics separately for {title_prefix}...")

    for metric_name in metrics:
        fig, ax = plt.subplots(figsize=(6, 4.5))
        
        model_data = results_dict[metric_name]
        
        for model_name, values in model_data.items():
            t_curr = times[:len(values)]
            
            # 获取样式参数 (color, linestyle 等)
            style_kwargs = color_map.get(model_name, {})
            
            # 绘图
            ax.plot(t_curr, values, **style_kwargs)
            
        # --- 风格统一化 ---
        
        # 1. 设置坐标轴标签
        ax.set_xlabel("Time (s)", fontsize=12)
        ax.set_ylabel(metric_name, fontsize=12)
        
        # 2. 图例：带边框，与上一张图保持一致
        # 如果图例遮挡了线条，可以改 loc='upper right' 或其他
        ax.legend(fontsize=10, loc='best', framealpha=1.0)
        
        # --- 修改点 2：删除 sns.despine()，强制显示全边框 ---
        # sns.despine(ax=ax)  <-- 删除这行
        
        # 强制确保上下左右边框都存在且为黑色
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_visible(True)
        
        # 3. 确保显示网格 (whitegrid 默认有，但这行是双重保险)
        ax.grid(True)

        # 4. 去掉标题
        ax.set_title("") 
        
        # --- 保存文件 ---
        safe_metric_name = metric_name.replace(" ", "_").replace("/", "-").replace("(", "").replace(")", "")
        save_name = f"{title_prefix}_{safe_metric_name}.pdf"
        
        plt.tight_layout()
        plt.savefig(save_name, format='pdf', bbox_inches='tight')
        plt.close(fig)
        
        print(f"  -> Saved {save_name}")
        
        
# def render_smpl_video(traj_data, output_file, 
#                       model_folder='./data/smplx', 
#                       reference_file='./data/CMU/144/144_10_stageii.npz',
#                       playback_fps=120): 


def render_smpl_video_final(traj_data, output_file, 
                            model_folder='./data/smplx', 
                            reference_file='./data/CMU/144/144_10_stageii.npz',
                            playback_fps=120,    # [Video Params] Control output video playback speed
                            data_fps=120,        # [Data Params] Actual data sampling rate (Ours fill 120, GT fill 30)
                            num_screenshots=7,   # [Screenshot Params] Automatically save 7 filmstrip screenshots
                            screenshot_file=None):
    
    # --- Just added trail config, does not affect original rendering ---
    trail_duration = 0.5             # Physics time: look back 0.5 seconds
    num_ghosts = 6                   # Number of ghosts
    ghost_color = [0.4, 0.4, 0.5]    # Ghost color (slightly darker, distinct from main)
    
    print(f"Rendering video (Motion Trails): {output_file} | Data FPS: {data_fps} | Playback FPS: {playback_fps}")
    
    import torch
    import smplx
    import open3d as o3d
    import imageio
    import numpy as np
    import os
    from tqdm import tqdm

    # Prepare screenshot filename
    if num_screenshots > 0 and screenshot_file is None:
        base, _ = os.path.splitext(output_file)
        screenshot_file = f"{base}_filmstrip.png"
    
    # Calculate screenshot indices
    screenshot_indices = set()
    if num_screenshots > 0:
        indices = np.linspace(0, traj_data.shape[0]-1, num_screenshots, dtype=int)
        screenshot_indices = set(indices)
    captured_shots = []

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 1. Dimension Processing (Keep original)
    D_ORIGINAL_STATE = 135
    if traj_data.shape[1] > D_ORIGINAL_STATE:
        traj_data = traj_data[:, -D_ORIGINAL_STATE:]
    
    # 2. Load Betas (Keep original)
    if os.path.exists(reference_file):
        try:
            ref_data = np.load(reference_file)
            real_betas_np = ref_data['betas'][:10] 
            real_betas = torch.tensor(real_betas_np, dtype=torch.float32).to(DEVICE)
        except:
            real_betas = torch.zeros(10, dtype=torch.float32).to(DEVICE)
    else:
        real_betas = torch.zeros(10, dtype=torch.float32).to(DEVICE)

    # 3. Initialize SMPL-X (Keep original)
    try:
        model = smplx.SMPLX(model_path=model_folder, gender='neutral', use_pca=False, 
                            num_betas=10, num_expression_coeffs=10, flat_hand_mean=False).to(DEVICE)
    except Exception as e:
        print(f"SMPL-X Init Failed: {e}"); return

    # 4. Data Inference (Keep original)
    T = traj_data.shape[0]
    data_tensor = torch.tensor(traj_data, dtype=torch.float32).to(DEVICE)
    
    trans = data_tensor[:, :3]
    pose_6d = data_tensor[:, 3:].reshape(T, 22, 6) 
    
    pose_mat = rotation_6d_to_matrix(pose_6d) 
    pose_aa = robust_matrix_to_axis_angle(pose_mat) 
    
    global_orient = pose_aa[:, 0:1, :]      
    body_pose     = pose_aa[:, 1:, :].reshape(T, 63)
    betas = real_betas.unsqueeze(0).repeat(T, 1)
    
    jaw   = torch.zeros((T, 3), device=DEVICE)
    leye  = torch.zeros((T, 3), device=DEVICE)
    reye  = torch.zeros((T, 3), device=DEVICE)
    expr  = torch.zeros((T, 10), device=DEVICE)
    lhand = model.left_hand_mean.reshape(1, 45).repeat(T, 1).to(DEVICE)
    rhand = model.right_hand_mean.reshape(1, 45).repeat(T, 1).to(DEVICE)

    with torch.no_grad():
        output = model(betas=betas, expression=expr, global_orient=global_orient, body_pose=body_pose, 
                       jaw_pose=jaw, leye_pose=leye, reye_pose=reye, left_hand_pose=lhand, right_hand_pose=rhand, 
                       transl=trans, return_verts=True)
    
    verts = output.vertices.cpu().numpy()
    faces = model.faces
    
    # 5. Open3D Visualization Settings (Strictly keep original)
    vis = o3d.visualization.Visualizer()
    vis.create_window(width=1088, height=1088, visible=False)
    
    # Main Mesh
    main_mesh = o3d.geometry.TriangleMesh()
    main_mesh.triangles = o3d.utility.Vector3iVector(faces)
    main_mesh.vertices = o3d.utility.Vector3dVector(verts[0])
    main_mesh.compute_vertex_normals()
    main_mesh.paint_uniform_color([0.65, 0.65, 0.7]) # [Confirm consistent color]
    vis.add_geometry(main_mesh)
    
    # --- [New] Ghost Object Pool (Independent of main character, does not affect scene center) ---
    window_frames = int(trail_duration * data_fps) 
    if window_frames < 1: window_frames = 1
    
    ghost_pool = []
    # Place at infinity, does not affect BBox calculation
    dummy_pos = np.ones((verts.shape[1], 3), dtype=np.float64) * 99999.0
    
    for i in range(num_ghosts):
        gm = o3d.geometry.TriangleMesh()
        gm.triangles = o3d.utility.Vector3iVector(faces)
        gm.vertices = o3d.utility.Vector3dVector(dummy_pos)
        gm.compute_vertex_normals()
        gm.paint_uniform_color(ghost_color)
        vis.add_geometry(gm, reset_bounding_box=False) # Key: Prevent reset view when adding
        ghost_pool.append(gm)
    
    # ---------------------------------------------------------

    # Environment Settings (Strictly keep original)
    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.1, 0.1, 0.1]) 
    opt.light_on = True
    
    # Camera Settings (Strictly keep original)
    ctr = vis.get_view_control()
    # Use full vertices to calculate BBox, ensuring center point is exactly same as original code
    bbox = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(verts.reshape(-1, 3))).get_axis_aligned_bounding_box()
    center = bbox.get_center()
    ctr.set_lookat(center); ctr.set_up([0, 0, 1]); ctr.set_front([0, -1, 0]); ctr.set_zoom(0.8)
    

    # [Modification 1] Ensure main character is in scene before loop starts
    vis.add_geometry(main_mesh, reset_bounding_box=False)

    # 6. Frame-by-frame Rendering
    images = []
    for t in tqdm(range(T), desc=f"Rendering"):
        # [Critical Modification 2] Remove main character from scene at start of each frame
        # This is to re-add it later, forcing it to be drawn last
        vis.remove_geometry(main_mesh, reset_bounding_box=False)

        # --- A. Update Ghosts (Keep original) ---
        start_t = max(0, t - window_frames)
        ghost_indices = np.linspace(start_t, t, num_ghosts, endpoint=False, dtype=int)
        ghost_indices = ghost_indices[ghost_indices != t]
        
        for i, gm in enumerate(ghost_pool):
            if i < len(ghost_indices):
                idx = ghost_indices[i]
                gm.vertices = o3d.utility.Vector3dVector(verts[idx])
                gm.compute_vertex_normals()
            else:
                gm.vertices = o3d.utility.Vector3dVector(dummy_pos)
            # Update ghost geometry
            vis.update_geometry(gm)
            

        # --- B. Update Main Character Data (Keep original) ---
        main_mesh.vertices = o3d.utility.Vector3dVector(verts[t])
        main_mesh.compute_vertex_normals()
        
        # [Critical Modification 3] Re-add main character to scene after updating all ghosts
        # This step guarantees main character is submitted to render queue last
        vis.add_geometry(main_mesh, reset_bounding_box=False)
        
        # First frame force refresh view (Keep original)
        if t == 0: 
            vis.poll_events()
            vis.update_renderer()
            ctr.set_lookat(center)
            
        vis.poll_events()
        vis.update_renderer()
        
        img = vis.capture_screen_float_buffer(True)
        img_uint8 = (np.asarray(img)*255).astype(np.uint8)
        images.append(img_uint8)
        
        # Collect screenshots
        if num_screenshots > 0 and t in screenshot_indices:
            captured_shots.append(img_uint8)
        
    vis.destroy_window()
    
    # Save Video
    try:
        imageio.mimsave(output_file, images, fps=playback_fps)
        print(f"Video saved: {output_file}")
    except Exception as e:
        print(f"Video save failed: {e}")

    # Save Filmstrip
    if num_screenshots > 0 and len(captured_shots) > 0:
        try:
            filmstrip = np.concatenate(captured_shots, axis=1)
            imageio.imwrite(screenshot_file, filmstrip)
            print(f"Filmstrip saved: {screenshot_file}")
        except Exception as e:
            print(f"Filmstrip save failed: {e}")


# ==============================================================================
# 5. Evaluation Wrapper
# ==============================================================================


def evaluate_all_models(extra_models_dict, val_loader, gt_times, gen_times_high_fps):
    DEVICE = gt_times.device 
    
    # Initialize FK Layer
    smpl_path = './data' 
    if not os.path.exists(smpl_path):
        print(f"Error: SMPL path {smpl_path} missing."); return {}, None, {}
    fk_layer = FastRigidFK(smpl_path, DEVICE)
    
    # Time Alignment
    diff = torch.abs(gen_times_high_fps.unsqueeze(1) - gt_times.unsqueeze(0))
    matched_indices = torch.argmin(diff, dim=0).to(DEVICE)
    
    # Prepare Accumulators (To store sum of metrics over time)
    # Structure: results[model_name][metric_name] = Tensor(T)
    accumulators = {}
    for name in extra_models_dict.keys():
        accumulators[name] = {
            'MPJPE_Sum': None, 'MPJVE_Sum': None, 'Jitter_Sum': None, 'Count': 0
        }

    sliced_outputs_for_vis = {} 
    start_idx = 0
    
    print(f"Evaluating in streaming batches (GPU)...")

    # --- STREAMING LOOP ---
    # Iterate through DataLoader to avoid loading full dataset at once
    for batch_i, (gt_batch_cpu, _) in enumerate(val_loader):
        batch_size = gt_batch_cpu.shape[0]
        end_idx = start_idx + batch_size
        
        # 1. Move GT to GPU
        gt_batch = gt_batch_cpu.to(DEVICE)
        
        for name, full_gen_tensor in extra_models_dict.items():
            if start_idx >= full_gen_tensor.shape[0]: continue
            
            # 2. Slice Generator (Reference only, no copy)
            gen_batch_subset = full_gen_tensor[start_idx:end_idx]
            gen_batch_gpu = gen_batch_subset.to(DEVICE)
            
            # 3. Time Downsampling
            T_actual = gen_batch_gpu.shape[1]
            valid_indices = torch.clamp(matched_indices, max=T_actual-1)
            gen_batch_downsampled = gen_batch_gpu[:, valid_indices, :]
            
            # 4. Handle NaNs
            if torch.isnan(gen_batch_downsampled).any():
                gen_batch_downsampled[torch.isnan(gen_batch_downsampled)] = 0.0

            # 5. Compute Metrics (GPU)
            batch_res = compute_batch_metrics(gen_batch_downsampled, gt_batch, fk_layer)
            
            # 6. Accumulate Sums
            acc = accumulators[name]
            if acc['MPJPE_Sum'] is None:
                # First batch: initialize tensors
                acc['MPJPE_Sum'] = batch_res['MPJPE_Sum']
                acc['MPJVE_Sum'] = batch_res['MPJVE_Sum']
                acc['Jitter_Sum'] = batch_res['Jitter_Sum']
            else:
                # Subsequent batches: add to sum
                acc['MPJPE_Sum'] += batch_res['MPJPE_Sum']
                acc['MPJVE_Sum'] += batch_res['MPJVE_Sum']
                acc['Jitter_Sum'] += batch_res['Jitter_Sum']
            
            acc['Count'] += batch_size

            # Save first batch for visualization
            if batch_i == 0:
                sliced_outputs_for_vis[name] = gen_batch_gpu 
                
        start_idx += batch_size
        # Cleanup batch memory
        del gt_batch, gen_batch_gpu, gen_batch_downsampled
        import gc; gc.collect(); torch.cuda.empty_cache()

    # --- Final Averaging ---
    final_results = {'MPJPE': {}, 'MPJVE': {}, 'Jitter': {}}
    
    print("\nEvaluation Complete:")
    for name, acc in accumulators.items():
        count = acc['Count']
        if count == 0: continue
        
        # Calculate Mean over Dataset (Sum / Count) -> Result is (T,)
        mpjpe_seq = (acc['MPJPE_Sum'] / count).tolist()
        mpjve_seq = (acc['MPJVE_Sum'] / count).tolist()
        jitter_seq = (acc['Jitter_Sum'] / count).tolist()
        
        final_results['MPJPE'][name] = mpjpe_seq
        final_results['MPJVE'][name] = mpjve_seq
        final_results['Jitter'][name] = jitter_seq
        
        avg_mpjpe = sum(mpjpe_seq) / len(mpjpe_seq)
        avg_mpjve = sum(mpjve_seq) / len(mpjve_seq)
        print(f"  -> {name}: Avg MPJPE={avg_mpjpe:.2f}mm, MPJVE={avg_mpjve:.2f}")

    # Reconstruct 1 batch of GT for visualization
    first_batch_gt = next(iter(val_loader))[0].to(DEVICE)

    return final_results, first_batch_gt, sliced_outputs_for_vis


# ==============================================================================
# 6. Main Execution
# ==============================================================================
if __name__ == "__main__":
    # Ensure output file path is defined
    try:
        d = torch.load(OUT)
        train_d, val_d = d['train_data'], d['val_data']
        if 'train_data' in d: del d['train_data']
        del d
        gc.collect()
        torch.cuda.empty_cache()
    except NameError:
        print("Error: OUTPUT_FILE (OUT) not defined in context."); exit()

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Time Config
    T_DENSE_GT = val_d['observation_times'].to(DEVICE) 
    GEN_FPS = 120
    duration = T_DENSE_GT[-1] - T_DENSE_GT[0]
    num_gen_steps = int(duration * GEN_FPS) + 1 
    T_GEN_HIGH = torch.linspace(T_DENSE_GT[0], T_DENSE_GT[-1], steps=num_gen_steps).to(DEVICE)

    print(f"Config: GT FPS=30 ({len(T_DENSE_GT)} frames) | Gen FPS={GEN_FPS} ({len(T_GEN_HIGH)} frames)")

    val_l = DataLoader(TensorDataset(val_d['trajectories'], val_d['conditions']), batch_size=256, shuffle=False)

    # Collect pre-generated model outputs from local variables
    extra_models = {}
    if 'generated_paths_ODE_linear' in locals(): extra_models['CSI'] = generated_paths_ODE_linear
    if 'generated_paths_ODE_spline' in locals(): extra_models['SSCI'] = generated_paths_ODE_spline
    
    del generated_paths_ODE_linear
    del generated_paths_ODE_spline
    torch.cuda.empty_cache()
    gc.collect()

    if not extra_models:
        print("No model outputs found in variables (generated_paths_ODE_...).")
    else:
        print("\n=== Running Evaluation (New Metrics) ===")

        # Now it is safe to run evaluation
        metrics, true_paths, high_fps_outs = evaluate_all_models(extra_models, val_l, T_DENSE_GT, T_GEN_HIGH)
        
        # Plotting
        cmap = plt.cm.get_cmap('tab10', len(extra_models))
        STYLE_CONFIG = {
            # --- 核心方法 (Our Method) ---
            "SSCI": {
                "color": "#D9363E",  # [Pyro] 绯红 (Klee Red) - 醒目但不刺眼
                "lw": 2.0,           # 线条最粗
                "linestyle": "-",    # 实线
                "zorder": 100,       # 永远在最顶层 (防止被遮挡)
                "alpha": 1.0,        # 不透明
                "label": "SSCI (Ours)"
            },
            
            # --- 主要对比方法 (Strong Baseline) ---
            "CSI": {
                "color": "#2E5EAA",  # [Hydro] 深宝石蓝 (Eula Blue) - 沉稳
                "lw": 2.0,           # 线条次粗
                "linestyle": "--",    # 实线
                "zorder": 90,        # 仅次于 SSCI
                "alpha": 1.0,
                "label": "CSI"
            },

            # --- 神经网络基线 (Neural Baselines) ---
            "LatentODEVaE": {       # 注意：匹配您代码中的键名(含空格)
                "color": "#56A36C",  # [Dendro] 草神绿 (Nahida Green) - 清新
                "lw": 2.0,           # 标准线宽
                "linestyle": "--",   # 虚线 (区分于 Ours)
                "zorder": 10,        # 后景
                "alpha": 1.0,
                "label": "LatentODEVaE"
            },
            
            # "LatentSDEVaE": {
            #     "color": "#7E57C2",  # [Electro] 雷电紫 (Raiden Purple) - 优雅
            #     "lw": 1.5,
            #     "linestyle": "--",
            #     "zorder": 10,
            #     "alpha": 1.0,
            #     "label": "LatentSDEVaE"
            # },
            
            "ConditionalTransformer": {
                "color": "#E09F3E",  # [Geo] 岩金 (Zhongli Amber) - 明亮
                "lw": 2.0,
                "linestyle": "--",   # 点划线 (增加区分度)
                "zorder": 10,
                "alpha": 1.0,
                "label": "ConditionalTransformer"
            },

            # --- 传统/非参基线 (Traditional/Non-Parametric) ---
            "GaussianProcess": {
                "color": "#7E57C2",  # [Electro] 雷电紫 (Raiden Purple) - 优雅 "#26A69A",  # [Anemo] 魈青 (Xiao Teal) - 高级灰调
                "lw": 2.0,
                "linestyle": "--",    # 点线 (表示非参的不确定性或不同性质)
                "zorder": 5,         # 最底层
                "alpha": 1.0,
                "label": "GaussianProcess"
            }
        }
        plot_comparative_metrics(metrics, T_DENSE_GT.cpu().numpy(), "Model Comparison", STYLE_CONFIG)

        # Rendering
        print("\n=== Rendering Visuals ===")
        # ------------------- 从这里开始修改 -------------------
        res_dir = Path("amass_vis_results")
        res_dir.mkdir(exist_ok=True)
        
        # 1. 在这里定义您想画的样本索引 (Indices)
        # 例如：画第0个, 第10个, 第50个样本 (请确保不超过验证集总长度)
        target_indices = [0, 100, 350]
        print(len(val_d['trajectories']))
        
        print(f"Target indices to render: {target_indices}")

        for idx in target_indices:
            print(f"--- Rendering Sample Index: {idx} ---")
            
            # A. 渲染 Ground Truth (30 FPS)
            # 注意：从 val_d['trajectories'] 直接取全量数据的第 idx 个
            gt_traj = val_d['trajectories'][idx].cpu().numpy()
            render_smpl_video_final(
                gt_traj, 
                str(res_dir / f"sample_{idx}_ground_truth_30fps.mp4"), 
                data_fps=30, 
                playback_fps=30
            )
            
            # B. 渲染所有生成模型 (120 FPS Slow-Mo)
            # 注意：从 extra_models 直接取全量数据的第 idx 个，不要用 high_fps_outs
            for name, full_tensor in extra_models.items():
                if idx >= full_tensor.shape[0]:
                    print(f"Skipping {name} for idx {idx} (out of bounds)")
                    continue
                    
                gen_traj = full_tensor[idx].cpu().numpy()
                render_smpl_video_final(
                    gen_traj, 
                    str(res_dir / f"sample_{idx}_{name}_120fps_slowmo.mp4"), 
                    data_fps=120, 
                    playback_fps=60 # 60fps播放实现2倍慢放，或者改回30fps实现4倍慢放
                )
        
        print("All Done.")


Config: GT FPS=30 (91 frames) | Gen FPS=120 (361 frames)

=== Running Evaluation (New Metrics) ===
[FastRigidFK] Loading SMPL-X from ./data...
[FastRigidFK] Skeleton extracted successfully.
Evaluating in streaming batches (GPU)...

Evaluation Complete:
  -> CSI: Avg MPJPE=94.10mm, MPJVE=5.88
  -> SSCI: Avg MPJPE=94.77mm, MPJVE=5.84
Plotting 3 metrics separately for Model Comparison...
  -> Saved Model Comparison_MPJPE.pdf
  -> Saved Model Comparison_MPJVE.pdf


C:\Users\Administrator\AppData\Local\Temp\ipykernel_32416\619499875.py:714: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab10', len(extra_models))


  -> Saved Model Comparison_Jitter.pdf

=== Rendering Visuals ===
360
Target indices to render: [0, 10, 100, 350]
--- Rendering Sample Index: 0 ---
Rendering video (Motion Trails): amass_vis_results\sample_0_ground_truth_30fps.mp4 | Data FPS: 30 | Playback FPS: 30


Rendering: 100%|██████████| 91/91 [00:04<00:00, 18.47it/s]


Video saved: amass_vis_results\sample_0_ground_truth_30fps.mp4
Filmstrip saved: amass_vis_results\sample_0_ground_truth_30fps_filmstrip.png
Rendering video (Motion Trails): amass_vis_results\sample_0_CSI_120fps_slowmo.mp4 | Data FPS: 120 | Playback FPS: 60


...

Video saved: amass_vis_results\sample_350_ground_truth_30fps.mp4
Filmstrip saved: amass_vis_results\sample_350_ground_truth_30fps_filmstrip.png
Rendering video (Motion Trails): amass_vis_results\sample_350_CSI_120fps_slowmo.mp4 | Data FPS: 120 | Playback FPS: 60


Rendering: 100%|██████████| 361/361 [00:19<00:00, 18.21it/s]


Video saved: amass_vis_results\sample_350_CSI_120fps_slowmo.mp4
Filmstrip saved: amass_vis_results\sample_350_CSI_120fps_slowmo_filmstrip.png
Rendering video (Motion Trails): amass_vis_results\sample_350_SSCI_120fps_slowmo.mp4 | Data FPS: 120 | Playback FPS: 60


Rendering: 100%|██████████| 361/361 [00:19<00:00, 18.31it/s]


Video saved: amass_vis_results\sample_350_SSCI_120fps_slowmo.mp4
Filmstrip saved: amass_vis_results\sample_350_SSCI_120fps_slowmo_filmstrip.png
All Done.
